<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# LogiTrack — Supply Chain Analytics
## Notebook 3 — SQL Analytics, KPIs & Performance
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Prérequis** | `logitrack_analytics.csv` dans `SAVE_PATH` (produit par NB2) |
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), pandas, matplotlib |
| **Durée estimée** | 3h à 4h |

> **5 questions business** auxquelles ce notebook répond :
> 1. Quels **corridors** concentrent les retards et pourquoi ?
> 2. Quels **transporteurs** performent le mieux en fiabilité × coût ?
> 3. Le taux de breach **s'améliore-t-il** dans le temps ?
> 4. À quelle **heure et quel jour** le volume est-il le plus élevé ?
> 5. Quel est le **coût financier réel** des retards par corridor et transporteur ?

---
## 0. Mise en place de l'environnement

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import duckdb
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

In [ ]:
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack/data/'
clean_path = f'{SAVE_PATH}logitrack_analytics.csv'

# Fallback GitHub si le fichier NB2 n'est pas disponible
if not os.path.exists(clean_path):
    clean_path = BASE_URL + 'logitrack_analytics.csv'
    print('⚠️  logitrack_analytics.csv non trouvé en local — chargement depuis GitHub')
else:
    print(f'✅ Fichier analytique trouvé : {clean_path}')

conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE livraisons    AS SELECT * FROM read_csv_auto('{clean_path}',  timestampformat='%Y-%m-%d');
    CREATE TABLE transporteurs AS SELECT * FROM read_csv_auto('{BASE_URL}transporteurs.csv');
    CREATE TABLE entrepots     AS SELECT * FROM read_csv_auto('{BASE_URL}entrepots.csv');
    CREATE TABLE routes        AS SELECT * FROM read_csv_auto('{BASE_URL}routes.csv');
    CREATE TABLE clients       AS SELECT * FROM read_csv_auto('{BASE_URL}clients.csv');
""")

n = conn.execute('SELECT COUNT(*) FROM livraisons').fetchone()[0]
print(f'✅ {n:,} livraisons chargées dans DuckDB')

%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

---
## Étape 1 — KPIs globaux

### MÉTHODE
Une seule requête pour tous les KPIs globaux — garantit la cohérence des chiffres (même filtre, même instant d'exécution). `TRY_CAST(csat AS DOUBLE)` protège contre les valeurs non numériques résiduelles. `NULLIF(x, 0)` protège contre les divisions par zéro.

In [ ]:
%%sql df_kpi_global <<
SELECT
    COUNT(*)                                                          AS total_livraisons,
    SUM(sla_breach)                                                   AS nb_breach,
    ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)                     AS taux_breach_pct,
    ROUND(AVG(TRY_CAST(csat AS DOUBLE)), 2)                          AS csat_moyen,
    SUM(escalade)                                                     AS nb_escalades,
    ROUND(SUM(escalade) * 100.0 / COUNT(*), 1)                       AS taux_escalade_pct,
    ROUND(SUM(penalite_fcfa), 0)                                      AS total_penalites_fcfa,
    ROUND(SUM(cout_transport_fcfa), 0)                                AS total_cout_transport,
    ROUND(AVG(CASE WHEN sla_breach = 1 THEN delai_jours END), 1)     AS delai_moy_retard_j,
    SUM(CASE WHEN statut = 'Perdu' THEN 1 ELSE 0 END)                AS nb_colis_perdus,
    COUNT(DISTINCT transporteur_id)                                   AS nb_transporteurs,
    COUNT(DISTINCT pays_origine || pays_destination)                  AS nb_corridors
FROM livraisons
WHERE transporteur_actif = 1

> **INTERPRÉTATION — KPIs de référence :**
>
> | KPI | Valeur | Lecture |
> |---|---|---|
> | Taux breach | **~39%** | 4 livraisons sur 10 dépassent le SLA |
> | CSAT moyen | **~3.65/5** | Satisfaction dégradée par les retards |
> | Taux escalade | **~6%** | Tickets remontés au management |
> | Délai moyen retard | **~6.3j** | Retard structurel, pas marginal |
>
> **MÉTIER :** Le ratio pénalités / coût transport révèle l'ampleur du problème. Même si les pénalités semblent faibles en valeur absolue, leur accumulation sur 15 000 livraisons représente un manque à gagner significatif — sans compter l'impact indirect sur la rétention des clients Grands Comptes.

---
## Étape 2 — Analyse des corridors avec RANK()

### MÉTHODE
`RANK() OVER (ORDER BY taux_breach DESC)` classe les corridors sans réduire le nombre de lignes. `CONCAT` (ou `||` en DuckDB) crée un libellé lisible. On joint `routes` pour récupérer `risque_douanier` et `distance_km` — deux variables explicatives clés du retard.

In [ ]:
%%sql df_corridors <<
SELECT
    l.pays_origine || ' → ' || l.pays_destination                    AS corridor,
    r.risque_douanier,
    r.distance_km,
    r.nb_points_controle,
    COUNT(l.livraison_id)                                             AS nb_livraisons,
    ROUND(SUM(l.sla_breach) * 100.0 / COUNT(*), 1)                   AS taux_breach_pct,
    ROUND(AVG(CASE WHEN l.sla_breach = 1
              THEN l.delai_jours END), 1)                            AS delai_moy_retard_j,
    ROUND(SUM(l.penalite_fcfa), 0)                                    AS penalites_fcfa,
    ROUND(AVG(TRY_CAST(l.csat AS DOUBLE)), 2)                        AS csat_moyen,
    SUM(l.escalade)                                                   AS nb_escalades,
    RANK() OVER (ORDER BY SUM(l.sla_breach) * 100.0
                          / COUNT(*) DESC)                           AS rang_risque
FROM livraisons l
JOIN routes r ON l.route_id = r.route_id
WHERE l.transporteur_actif = 1
GROUP BY l.pays_origine, l.pays_destination,
         r.risque_douanier, r.distance_km, r.nb_points_controle
ORDER BY rang_risque

> **INTERPRÉTATION :**
> - Les corridors impliquant **Mali** et **Burkina Faso** dominent le top des breaches — `risque_douanier = Élevé` + transporteurs moins fiables (TRP05, TRP06, TRP12)
> - Les corridors `Faible` risque douanier (Côte d'Ivoire ↔ Ghana, Sénégal ↔ Côte d'Ivoire) ont des taux breach nettement inférieurs
> - La corrélation **risque douanier × taux breach** sera une feature importante en NB5
>
> **MÉTIER :** Un corridor à risque douanier élevé ne doit pas nécessairement être abandonné — mais les SLA contractuels doivent être recalibrés pour refléter la réalité opérationnelle. Promettre 5 jours sur un corridor qui prend structurellement 8 jours est une promesse impossible à tenir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Graphique 1 — Taux breach par corridor (top 10)
top10 = df_corridors.head(10).sort_values('taux_breach_pct')
colors_corr = [COLORS['danger'] if r == 'Élevé' else
               COLORS['warning'] if r == 'Moyen' else
               COLORS['secondary'] for r in top10['risque_douanier']]
axes[0].barh(top10['corridor'], top10['taux_breach_pct'], color=colors_corr)
axes[0].axvline(df_kpi_global['taux_breach_pct'].iloc[0],
                color=COLORS['neutral'], linestyle='--', linewidth=1.5,
                label=f"Moyenne {df_kpi_global['taux_breach_pct'].iloc[0]}%")
axes[0].set_title('Taux breach par corridor (top 10)', fontweight='bold')
axes[0].set_xlabel('Taux breach (%)')
axes[0].legend(fontsize=9)

# Graphique 2 — Scatter distance × taux breach coloré par risque
risque_colors = {'Faible': COLORS['secondary'],
                 'Moyen':  COLORS['warning'],
                 'Élevé':  COLORS['danger']}
for risque, grp in df_corridors.groupby('risque_douanier'):
    axes[1].scatter(grp['distance_km'], grp['taux_breach_pct'],
                    color=risque_colors.get(risque, COLORS['neutral']),
                    label=risque, s=grp['nb_livraisons']/8, alpha=0.75)
axes[1].set_title('Distance × Taux breach (taille = volume)', fontweight='bold')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Taux breach (%)')
axes[1].legend(title='Risque douanier')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_corridors.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_corridors.png')

---
## Étape 3 — Matrice performance transporteurs

### MÉTHODE
On construit une matrice à 3 dimensions — fiabilité (taux breach), coût, CSAT — pour chaque transporteur actif. `NTILE(3)` segmente les transporteurs en 3 tiers égaux selon chaque dimension. `RANK()` dans une CTE permet d'appliquer plusieurs classements dans la même requête.

In [ ]:
%%sql df_trp_perf <<
WITH perf AS (
    SELECT
        l.transporteur_id,
        t.nom                                                         AS transporteur,
        t.mode_transport,
        COUNT(l.livraison_id)                                         AS nb_livraisons,
        ROUND(SUM(l.sla_breach) * 100.0 / COUNT(*), 1)               AS taux_breach_pct,
        ROUND(AVG(TRY_CAST(l.csat AS DOUBLE)), 2)                    AS csat_moyen,
        ROUND(AVG(l.duree_reelle_jours), 1)                          AS duree_moy_j,
        ROUND(AVG(l.cout_transport_fcfa), 0)                         AS cout_moy_fcfa,
        ROUND(SUM(l.penalite_fcfa), 0)                               AS total_penalites,
        ROUND(SUM(l.penalite_fcfa) * 100.0
              / NULLIF(SUM(l.cout_transport_fcfa), 0), 2)            AS ratio_penalite_cout_pct
    FROM livraisons l
    JOIN transporteurs t ON l.transporteur_id = t.transporteur_id
    WHERE l.transporteur_actif = 1
    GROUP BY l.transporteur_id, t.nom, t.mode_transport
)
SELECT
    transporteur,
    mode_transport,
    nb_livraisons,
    taux_breach_pct,
    csat_moyen,
    duree_moy_j,
    cout_moy_fcfa,
    total_penalites,
    ratio_penalite_cout_pct,
    RANK() OVER (ORDER BY taux_breach_pct ASC)                        AS rang_fiabilite,
    RANK() OVER (ORDER BY csat_moyen DESC)                            AS rang_csat,
    RANK() OVER (ORDER BY cout_moy_fcfa ASC)                         AS rang_cout,
    NTILE(3) OVER (ORDER BY taux_breach_pct ASC)                     AS tier_fiabilite
FROM perf
ORDER BY rang_fiabilite

> **INTERPRÉTATION — Matrice transporteurs :**
>
> | Transporteur | Taux breach | CSAT | Coût moy | Profil |
> |---|---|---|---|---|
> | TRP08 (AirFreight) | ~3% | 4.8 | Très élevé | Champion — hors budget |
> | TRP07 (TransOuest) | ~7% | 4.5 | Élevé | Meilleur rapport qualité/coût |
> | TRP03 (CamLog) | ~9% | 4.3 | Moyen | Fiable sur corridors Cameroun |
> | TRP12 (NordSud) | ~65% | 2.9 | Bas | Moins cher, mais coûteux en pénalités |
>
> **MÉTIER :** `ratio_penalite_cout_pct` est l'indicateur décisif. TRP12 a un tarif bas mais un ratio pénalités/coût qui efface l'économie. Le coût réel d'une livraison TRP12 = tarif + pénalités + impact CSAT.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter CSAT × Taux breach (taille = volume, couleur = tier)
tier_colors = {1: COLORS['secondary'], 2: COLORS['warning'], 3: COLORS['danger']}
for _, row in df_trp_perf.iterrows():
    axes[0].scatter(row['taux_breach_pct'], row['csat_moyen'],
                    s=row['nb_livraisons']/4,
                    color=tier_colors.get(row['tier_fiabilite'], COLORS['neutral']),
                    alpha=0.75, zorder=5)
    axes[0].annotate(row['transporteur'].split(' ')[0],
                     (row['taux_breach_pct'], row['csat_moyen']),
                     textcoords='offset points', xytext=(6, 4), fontsize=8)
med_breach = df_trp_perf['taux_breach_pct'].median()
med_csat   = df_trp_perf['csat_moyen'].median()
axes[0].axvline(med_breach, color=COLORS['neutral'], linestyle='--', linewidth=1, alpha=0.7)
axes[0].axhline(med_csat,   color=COLORS['neutral'], linestyle='--', linewidth=1, alpha=0.7)
axes[0].text(med_breach+0.5, med_csat+0.05, 'Top Performers',
             color=COLORS['secondary'], fontsize=9, fontweight='bold')
axes[0].set_title('Matrice transporteurs : Fiabilité × Satisfaction', fontweight='bold')
axes[0].set_xlabel('Taux breach (%)')
axes[0].set_ylabel('CSAT moyen')

# Barres pénalités totales par transporteur
trp_sorted = df_trp_perf.sort_values('total_penalites', ascending=True)
axes[1].barh(trp_sorted['transporteur'],
             trp_sorted['total_penalites'] / 1000,
             color=[tier_colors.get(t, COLORS['neutral'])
                    for t in trp_sorted['tier_fiabilite']])
axes[1].set_title('Pénalités totales par transporteur (k FCFA)', fontweight='bold')
axes[1].set_xlabel('Pénalités (k FCFA)')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_transporteurs.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_transporteurs.png')

---
## Étape 4 — Tendance mensuelle avec LAG()

### MÉTHODE
`LAG(taux_breach_pct) OVER (ORDER BY mois)` retourne le taux du mois précédent. La variation `taux_actuel - taux_precedent` mesure si la situation s'améliore ou se dégrade. Un écart-type élevé des variations signale une instabilité opérationnelle — pas seulement un problème de niveau mais de régularité.

In [ ]:
%%sql df_mensuel <<
WITH mensuel AS (
    SELECT
        strftime(date_creation, '%Y-%m')                              AS mois,
        COUNT(*)                                                      AS nb_livraisons,
        SUM(sla_breach)                                               AS nb_breach,
        ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)                 AS taux_breach_pct,
        ROUND(AVG(TRY_CAST(csat AS DOUBLE)), 2)                      AS csat_moy,
        ROUND(SUM(penalite_fcfa), 0)                                  AS penalites_fcfa,
        SUM(escalade)                                                 AS nb_escalades
    FROM livraisons
    WHERE transporteur_actif = 1
    GROUP BY strftime(date_creation, '%Y-%m')
)
SELECT
    mois,
    nb_livraisons,
    nb_breach,
    taux_breach_pct,
    csat_moy,
    penalites_fcfa,
    nb_escalades,
    LAG(taux_breach_pct) OVER (ORDER BY mois)                        AS breach_mois_prec,
    ROUND(
        taux_breach_pct
        - LAG(taux_breach_pct) OVER (ORDER BY mois)
    , 1)                                                             AS variation_breach,
    LAG(nb_livraisons) OVER (ORDER BY mois)                         AS vol_mois_prec,
    ROUND(
        (nb_livraisons - LAG(nb_livraisons) OVER (ORDER BY mois))
        * 100.0
        / NULLIF(LAG(nb_livraisons) OVER (ORDER BY mois), 0)
    , 1)                                                             AS variation_vol_pct
FROM mensuel
ORDER BY mois

> **INTERPRÉTATION — Tendance mensuelle :**
> - Le taux de breach oscille autour de 39% **sans tendance baissière claire** sur 24 mois — confirmation que le problème est structurel
> - Les pics de novembre-janvier (+5 à +8 points vs moyenne) coïncident avec les pics de volume
> - La corrélation `variation_vol_pct` × `variation_breach` est positive — plus le volume monte, plus le taux de breach monte
>
> **MÉTIER :** L'absence d'amélioration sur 24 mois signifie qu'aucune action corrective n'a eu d'effet durable. Le système d'alerte ML du NB5 vise précisément à casser cette tendance en permettant des interventions préventives ciblées.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Volume mensuel
axes[0].bar(df_mensuel['mois'], df_mensuel['nb_livraisons'],
            color=COLORS['primary'], alpha=0.6, label='Volume')
axes[0].set_title('Volume mensuel de livraisons — LogiTrack 2022-2023',
                  fontweight='bold')
axes[0].set_ylabel('Nb livraisons')
axes[0].legend()

# Taux breach + variation
ax2b = axes[1].twinx()
axes[1].plot(df_mensuel['mois'], df_mensuel['taux_breach_pct'],
             color=COLORS['danger'], linewidth=2.5, marker='o', markersize=5,
             label='Taux breach')
axes[1].axhline(df_mensuel['taux_breach_pct'].mean(),
                color=COLORS['warning'], linestyle='--', linewidth=1.5,
                label=f"Moyenne {df_mensuel['taux_breach_pct'].mean():.1f}%")
ax2b.bar(df_mensuel['mois'],
         df_mensuel['variation_breach'].fillna(0),
         color=df_mensuel['variation_breach'].apply(
             lambda x: COLORS['danger'] if x > 0 else COLORS['secondary']),
         alpha=0.4, label='Variation')
axes[1].set_title('Taux de breach mensuel & variation vs mois précédent',
                  fontweight='bold')
axes[1].set_ylabel('Taux breach (%)')
ax2b.set_ylabel('Variation (pts)')
axes[1].legend(loc='upper left')
axes[1].tick_params(axis='x', rotation=45)
step = max(1, len(df_mensuel)//12)
axes[1].set_xticks(range(0, len(df_mensuel), step))
axes[1].set_xticklabels(df_mensuel['mois'].iloc[::step], rotation=45)

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_tendance_mensuelle.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_tendance_mensuelle.png')

---
## Étape 5 — Heatmap volume par jour × heure

### MÉTHODE
On extrait le jour de la semaine et l'heure depuis `date_enlevement`, puis on pivote avec `pandas.pivot_table()`. La heatmap révèle les créneaux à haute tension opérationnelle — utile pour planifier les équipes de dispatch et les capacités entrepôt.

In [ ]:
%%sql df_heatmap_raw <<
SELECT
    DAYOFWEEK(date_enlevement)                                        AS jour_num,
    heure_enlevement                                                  AS heure,
    COUNT(*)                                                          AS nb_livraisons,
    ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)                     AS taux_breach_pct
FROM livraisons
WHERE transporteur_actif = 1
  AND heure_enlevement IS NOT NULL
GROUP BY DAYOFWEEK(date_enlevement), heure_enlevement
ORDER BY jour_num, heure

In [ ]:
JOURS = {1: 'Lun', 2: 'Mar', 3: 'Mer', 4: 'Jeu', 5: 'Ven', 6: 'Sam', 0: 'Dim'}
df_heatmap_raw['jour_nom'] = df_heatmap_raw['jour_num'].map(JOURS)

# Pivot volume
pivot_vol = df_heatmap_raw.pivot_table(
    index='heure', columns='jour_nom', values='nb_livraisons'
).fillna(0).astype(int)
# Réordonner les jours
jours_ordre = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
pivot_vol = pivot_vol[[j for j in jours_ordre if j in pivot_vol.columns]]

# Top 3 créneaux
top3 = df_heatmap_raw.nlargest(3, 'nb_livraisons')
print('Top 3 créneaux de volume :')
for _, r in top3.iterrows():
    print(f'  {r["jour_nom"]} {int(r["heure"])}h : {int(r["nb_livraisons"])} livraisons '
          f'(breach {r["taux_breach_pct"]}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Heatmap volume
sns.heatmap(pivot_vol, cmap='Blues', annot=False,
            linewidths=0.3, ax=axes[0],
            cbar_kws={'label': 'Nb livraisons'})
axes[0].set_title('Volume enlèvements — Heure × Jour', fontweight='bold')
axes[0].set_ylabel('Heure')
axes[0].set_xlabel('')

# Pivot taux breach
pivot_breach = df_heatmap_raw.pivot_table(
    index='heure', columns='jour_nom', values='taux_breach_pct'
).fillna(0)
pivot_breach = pivot_breach[[j for j in jours_ordre if j in pivot_breach.columns]]
sns.heatmap(pivot_breach, cmap='YlOrRd', annot=False,
            linewidths=0.3, ax=axes[1],
            cbar_kws={'label': 'Taux breach (%)'})
axes[1].set_title('Taux breach — Heure × Jour', fontweight='bold')
axes[1].set_ylabel('')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_heatmap_volume.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_heatmap_volume.png')

> **INTERPRÉTATION :**
> - Le volume est concentré en **semaine de 8h à 17h** — LogiTrack est principalement B2B
> - Les **vendredis après-midi** ont un taux de breach plus élevé — enlèvements tardifs qui n'arrivent pas à destination avant le weekend
> - Les **lundis matins** voient un pic de volume (remise en route post-weekend) avec un taux de breach supérieur à la moyenne
>
> **MÉTIER :** Décourager les enlèvements du **vendredi après 14h** sur les corridors longs réduirait structurellement le taux de breach du weekend. Cette règle opérationnelle simple n'a pas besoin du ML pour être implémentée.

---
## Étape 6 — Coût financier du retard

### MÉTHODE
On distingue le **coût direct** (pénalités contractuelles) du **coût de réacheminement** (estimé à 15% du coût transport pour les livraisons en breach). La somme des deux donne le **coût total du retard** — plus pertinent pour la direction que le seul taux de breach.

In [ ]:
%%sql df_cout <<
WITH cout_par_corridor AS (
    SELECT
        pays_origine || ' → ' || pays_destination                     AS corridor,
        COUNT(*)                                                      AS nb_livraisons,
        SUM(sla_breach)                                               AS nb_breach,
        ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)                 AS taux_breach_pct,
        ROUND(SUM(penalite_fcfa), 0)                                  AS penalites_directes,
        ROUND(SUM(CASE WHEN sla_breach = 1
              THEN cout_transport_fcfa * 0.15 ELSE 0 END), 0)        AS cout_reacheminement_estime,
        ROUND(SUM(penalite_fcfa)
              + SUM(CASE WHEN sla_breach = 1
                    THEN cout_transport_fcfa * 0.15 ELSE 0 END), 0)  AS cout_total_retard,
        RANK() OVER (ORDER BY
            SUM(penalite_fcfa)
            + SUM(CASE WHEN sla_breach = 1
                  THEN cout_transport_fcfa * 0.15 ELSE 0 END) DESC)  AS rang_cout
    FROM livraisons
    WHERE transporteur_actif = 1
    GROUP BY pays_origine, pays_destination
)
SELECT *
FROM cout_par_corridor
ORDER BY rang_cout
LIMIT 10

> **INTERPRÉTATION — Coût financier du retard :**
> - Les corridors sahéliens (Mali, Burkina Faso) cumulent le plus haut taux de breach ET le coût total le plus élevé
> - Le coût de réacheminement estimé (15% du coût transport) peut dépasser les pénalités directes sur les corridors longs
>
> **MÉTIER :** Ce tableau est le plus impactant pour convaincre la direction d'investir dans le système d'alerte ML. Le retour sur investissement est simple : si le modèle prévient 20% des retards sur les corridors les plus coûteux, l'économie annuelle couvre largement le coût de développement.

---
## Étape 7 — Entrepôts goulots d'étranglement

### MÉTHODE
Un entrepôt goulot est celui dont le `taux_breach_depuis` (calculé sur les livraisons partant de cet entrepôt) dépasse significativement son `taux_ponctualite` déclaré. L'écart entre les deux révèle un problème opérationnel non documenté.

In [ ]:
%%sql df_entrepots <<
SELECT
    e.entrepot_id,
    e.nom                                                             AS entrepot,
    e.type,
    e.pays,
    e.taux_ponctualite                                                AS ponctualite_declaree,
    COUNT(l.livraison_id)                                             AS nb_departs,
    ROUND(SUM(l.sla_breach) * 100.0 / COUNT(*), 1)                   AS taux_breach_depuis,
    ROUND(AVG(l.delai_prep_heures), 1)                               AS delai_prep_moy_h,
    ROUND(e.taux_ponctualite * 100
          - (1 - SUM(l.sla_breach) * 1.0 / COUNT(*)) * 100, 1)      AS ecart_ponctualite,
    RANK() OVER (ORDER BY SUM(l.sla_breach) * 100.0
                          / COUNT(*) DESC)                           AS rang_risque
FROM livraisons l
JOIN entrepots e ON l.entrepot_origine_id = e.entrepot_id
WHERE l.transporteur_actif = 1
GROUP BY e.entrepot_id, e.nom, e.type, e.pays, e.taux_ponctualite
ORDER BY rang_risque

> **INTERPRÉTATION — Entrepôts :**
> - **Bamako Hub** et **Ouagadougou Hub** ont les taux de breach au départ les plus élevés — cohérent avec leur `taux_ponctualite` déclaré (0.75-0.77)
> - `delai_prep_moy_h` élevé sur ces hubs confirme que le retard commence avant même l'enlèvement
> - Les entrepôts Hub **Abidjan** et **Douala** performent bien malgré des volumes élevés
>
> **MÉTIER :** Réduire le délai de préparation sur Bamako et Ouagadougou de 13h à 8h permettrait d'absorber une partie du retard avant même que le transporteur intervienne. Une action managériale simple (process checklist au départ) peut avoir un impact mesurable sur le taux de breach.

---
## Export des datasets pour Power BI

In [ ]:
# Export des résultats analytiques pour Power BI
df_corridors.to_csv(f'{SAVE_PATH}logitrack_corridors.csv', index=False)
df_trp_perf.to_csv(f'{SAVE_PATH}logitrack_transporteurs_perf.csv', index=False)
df_mensuel.to_csv(f'{SAVE_PATH}logitrack_mensuel.csv', index=False)
df_cout.to_csv(f'{SAVE_PATH}logitrack_cout_retard.csv', index=False)
df_entrepots.to_csv(f'{SAVE_PATH}logitrack_entrepots_perf.csv', index=False)

print('Fichiers exportés ✅')
print(f'  logitrack_corridors.csv           → performance par corridor')
print(f'  logitrack_transporteurs_perf.csv  → matrice transporteurs')
print(f'  logitrack_mensuel.csv             → tendance mensuelle + LAG')
print(f'  logitrack_cout_retard.csv         → coût financier du retard')
print(f'  logitrack_entrepots_perf.csv      → goulots entrepôts')
print(f'\n📁 Localisation : {SAVE_PATH}')

---
## Bilan du Notebook 3

### Réponses aux 5 questions business

| Question | Réponse |
|---|---|
| Corridors les plus problématiques | **Mali et Burkina Faso** — risque douanier Élevé + TRP12/05/06 |
| Meilleur rapport transporteur fiabilité/coût | **TRP07 (TransOuest)** et **TRP03 (CamLog)** |
| Tendance du taux de breach | **Stable à ~39%** — aucune amélioration sur 24 mois |
| Créneau de volume le plus élevé | **Lundi-vendredi 8h-17h** — B2B majoritaire |
| Coût réel du retard | **Pénalités + 15% coût transport** — corridors sahéliens en tête |

### Patterns SQL maîtrisés

| Pattern | Utilisation |
|---|---|
| `RANK() OVER (ORDER BY)` | Classement corridors, transporteurs, entrepôts |
| `NTILE(3) OVER` | Segmentation transporteurs en tiers de performance |
| `LAG() OVER (ORDER BY mois)` | Variation mensuelle taux breach et volume |
| `TRY_CAST(csat AS DOUBLE)` | Cast sécurisé sur colonne avec nulls |
| `NULLIF(x, 0)` | Protection division par zéro |
| `strftime(date, '%Y-%m')` | Extraction mois depuis une date |
| `DAYOFWEEK(date)` | Extraction jour de la semaine |
| `pandas.pivot_table()` | Heatmap volume et breach |

**Fichiers exportés dans `SAVE_PATH` :** 5 CSV prêts pour Power BI

**Pour le NB4 :** connecter ces 5 fichiers + `logitrack_analytics.csv` dans Power BI, créer les mesures DAX et construire le dashboard 5 pages.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.